In [1]:
import pandas as pd


In [2]:
data = pd.read_csv('../data/filtered_gen_1M.csv')

In [3]:
data.head()

,series_id,name,units,f,copyright,source,iso3166,lat,lon,geography,start,end,last_updated,latlon,period,value
0,ELEC.GEN.WND-WSC-1.M,Net generation : wind : West South Central (to...,thousand megawatthours,M,NaN,"EIA, U.S. Energy Information Administration",NaN,NaN,NaN,USA-AR+USA-LA+USA-OK+USA-TX,200102,202507,2025-09-22T22:39:56-04:00,NaN,202507,748.46786
1,ELEC.GEN.WND-WSC-1.M,Net generation : wind : West South Central (to...,thousand megawatthours,M,NaN,"EIA, U.S. Energy Information Administration",NaN,NaN,NaN,USA-AR+USA-LA+USA-OK+USA-TX,200102,202507,2025-09-22T22:39:56-04:00,NaN,202506,790.12045
2,ELEC.GEN.WND-WSC-1.M,Net generation : wind : West South Central (to...,thousand megawatthours,M,NaN,"EIA, U.S. Energy Information Administration",NaN,NaN,NaN,USA-AR+USA-LA+USA-OK+USA-TX,200102,202507,2025-09-22T22:39:56-04:00,NaN,202505,675.14392
3,ELEC.GEN.WND-WSC-1.M,Net generation : wind : West South Central (to...,thousand megawatthours,M,NaN,"EIA, U.S. Energy Information Administration",NaN,NaN,NaN,USA-AR+USA-LA+USA-OK+USA-TX,200102,202507,2025-09-22T22:39:56-04:00,NaN,202504,814.34525
4,ELEC.GEN.WND-WSC-1.M,Net generation : wind : West South Central (to...,thousand megawatthours,M,NaN,"EIA, U.S. Energy Information Administration",NaN,NaN,NaN,USA-AR+USA-LA+USA-OK+USA-TX,200102,202507,2025-09-22T22:39:56-04:00,NaN,202503,863.10946


In [4]:
data.shape

(174975, 16)

In [5]:
data.dtypes

series_id        object
name             object
units            object
f                object
copyright       float64
source           object
iso3166          object
lat             float64
lon             float64
geography        object
start             int64
end               int64
last_updated     object
latlon          float64
period            int64
value           float64
dtype: object

In [6]:
#Extract Year ands Month from Period

data['period'] = pd.to_datetime(data['period'], format = '%Y%m')
data['year'] = data['period'].dt.year
data['month'] = data['period'].dt.month
data['month_name'] = data.period.dt.month_name()

#Extract State from iso3166 format (last 2 characters)

data['state'] = data['iso3166'].str.split('-').str[1]


#Extract Fuel Type from Name
data['fuel'] = data['name'].str.split(':').str[1]

In [7]:
data.dtypes

series_id               object
name                    object
units                   object
f                       object
copyright              float64
source                  object
iso3166                 object
lat                    float64
lon                    float64
geography               object
start                    int64
end                      int64
last_updated            object
latlon                 float64
period          datetime64[ns]
value                  float64
year                     int32
month                    int32
month_name              object
state                   object
fuel                    object
dtype: object

In [8]:
data.fuel.unique().tolist()

[' wind ',
 ' wood and wood-derived fuels ',
 ' other biomass ',
 ' utility-scale photovoltaic ',
 ' utility-scale thermal ',
 ' all utility-scale solar ',
 ' petroleum liquids ',
 ' hydro-electric pumped storage ',
 ' geothermal ',
 ' other renewables (total) ',
 ' all fuels ',
 ' coal ',
 ' conventional hydroelectric ',
 ' natural gas ',
 ' petroleum coke ',
 ' other ',
 ' other gases ',
 ' nuclear ',
 ' all solar ']

In [9]:
#Delete Select Fuel Type Records

data = data.drop(data.loc[data['fuel'] == ' all solar '].index)
data = data.drop(data.loc[data['fuel'] == ' all utility-scale solar '].index)
data = data.drop(data.loc[data['fuel'] == ' utility-scale thermal '].index)
data = data.drop(data.loc[data['fuel'] == ' all solar '].index)


In [10]:
# Create Fuel Category Column (Renewable Energy or Non-Renewable Energy)
fuel_category_dict = {' wind ' : 'renewable energy',
 ' wood and wood-derived fuels ' : 'renewable energy',
 ' other biomass ' : 'renewable energy',
 ' utility-scale photovoltaic ' : 'renewable energy',
 ' petroleum liquids ' : 'non-renewable energy',
 ' hydro-electric pumped storage ': 'renewable energy',
 ' geothermal ' : 'renewable energy',
 ' other renewables (total) ' : 'renewable energy',
 ' all fuels ' : 'all energy',
 ' coal ' : 'non-renewable energy',
 ' conventional hydroelectric ' : 'renewable energy',
 ' natural gas ' : 'non-renewable energy',
 ' petroleum coke ' : 'non-renewable energy',
 ' other ' : 'non-renewable energy',
 ' other gases ' : 'non-renewable energy',
 ' nuclear ' : 'non-renewable energy'
}
data['fuel_category'] = data.fuel.map(fuel_category_dict)

In [11]:
# Create Revised Fuel
revised_fuel_dict = {' wind ' : 'wind',
 ' wood and wood-derived fuels ' : 'wood',
 ' other biomass ' : 'biomass',
 ' utility-scale photovoltaic ' : 'solar',
 ' petroleum liquids ' : 'petroleum',
 ' hydro-electric pumped storage ': 'hydroelectric',
 ' geothermal ' : 'geothermal',
 ' other renewables (total) ' : 'other',
 ' all fuels ' : 'all fuels',
 ' coal ' : 'coal',
 ' conventional hydroelectric ' : 'hydroelectric',
 ' natural gas ' : 'natural gas',
 ' petroleum coke ' : 'petroleum',
 ' other ' : 'other',
 ' other gases ' : 'other',
 ' nuclear ' : 'nuclear'
}
data['revised_fuel'] = data.fuel.map(revised_fuel_dict)

In [12]:
data.head()

,series_id,name,units,f,copyright,source,iso3166,lat,lon,geography,...,latlon,period,value,year,month,month_name,state,fuel,fuel_category,revised_fuel
0,ELEC.GEN.WND-WSC-1.M,Net generation : wind : West South Central (to...,thousand megawatthours,M,NaN,"EIA, U.S. Energy Information Administration",NaN,NaN,NaN,USA-AR+USA-LA+USA-OK+USA-TX,...,NaN,2025-07-01,748.46786,2025,7,July,NaN,wind,renewable energy,wind
1,ELEC.GEN.WND-WSC-1.M,Net generation : wind : West South Central (to...,thousand megawatthours,M,NaN,"EIA, U.S. Energy Information Administration",NaN,NaN,NaN,USA-AR+USA-LA+USA-OK+USA-TX,...,NaN,2025-06-01,790.12045,2025,6,June,NaN,wind,renewable energy,wind
2,ELEC.GEN.WND-WSC-1.M,Net generation : wind : West South Central (to...,thousand megawatthours,M,NaN,"EIA, U.S. Energy Information Administration",NaN,NaN,NaN,USA-AR+USA-LA+USA-OK+USA-TX,...,NaN,2025-05-01,675.14392,2025,5,May,NaN,wind,renewable energy,wind
3,ELEC.GEN.WND-WSC-1.M,Net generation : wind : West South Central (to...,thousand megawatthours,M,NaN,"EIA, U.S. Energy Information Administration",NaN,NaN,NaN,USA-AR+USA-LA+USA-OK+USA-TX,...,NaN,2025-04-01,814.34525,2025,4,April,NaN,wind,renewable energy,wind
4,ELEC.GEN.WND-WSC-1.M,Net generation : wind : West South Central (to...,thousand megawatthours,M,NaN,"EIA, U.S. Energy Information Administration",NaN,NaN,NaN,USA-AR+USA-LA+USA-OK+USA-TX,...,NaN,2025-03-01,863.10946,2025,3,March,NaN,wind,renewable energy,wind


In [13]:
#Filter out records where ISO1366 value is Nan. Those are regional rollups that are not needed because we are focusing on state.
data = data.loc[data['state'].notna()]

In [14]:
data.shape

(119499, 23)

In [15]:
#Filter data for 2015 to 2025, the last 10 years.
data_ge_2015 = data.loc[data['year'] >= 2015]

In [16]:
data_ge_2015.shape

(55518, 23)

In [17]:
#data_ge_2015.loc[data_ge_2015['name'].str.contains('solar')]

In [18]:
file_path = "C:/Users/marvi/Documents/DA15/Python/projects/capstone2025_marvin_short/data/state_fuel_data.csv"

data_ge_2015.to_csv(file_path, index=False)

In [19]:
data_ge_2015.columns

Index(['series_id', 'name', 'units', 'f', 'copyright', 'source', 'iso3166',
       'lat', 'lon', 'geography', 'start', 'end', 'last_updated', 'latlon',
       'period', 'value', 'year', 'month', 'month_name', 'state', 'fuel',
       'fuel_category', 'revised_fuel'],
      dtype='object')